# Agent Platform Colab bootstrap

CAGB-0/CAGB-1 one-tap mode: current public main is bound once to an exact SHA, then a no-model CPU provenance smoke writes a sanitized Drive bundle. No model or GPU execution.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys

repo = Path("/content/nyang-repo")
if repo.exists():
    shutil.rmtree(repo)
subprocess.run([
    "git", "clone", "--filter=blob:none", "--branch", "main",
    "--single-branch", "https://github.com/hanmiyoo10-alt/-.git", str(repo)
], check=True)
package_root = repo / "tools" / "agent-skill-orchestrator"
sys.path.insert(0, str(package_root))
from benchmarks.colab.request import make_request, make_runtime_request_id, resolve_checked_out_main_sha

REPOSITORY_SHA = resolve_checked_out_main_sha(repo)
REQUEST_ID = make_runtime_request_id()
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", REPOSITORY_SHA], check=True)
print(json.dumps({"repository_sha": REPOSITORY_SHA, "request_id": REQUEST_ID}, sort_keys=True))


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from benchmarks.colab.bootstrap import run_bootstrap, validate_bundle
request = make_request(REQUEST_ID, REPOSITORY_SHA)
receipt = run_bootstrap(request, repo, Path("/content/drive/MyDrive"))
bundle = Path("/content/drive/MyDrive") / receipt["drive_handoff_relative_path"]
validation = validate_bundle(bundle)
print(json.dumps(validation, sort_keys=True))
